# 06 · Submission (Kaggle, GPU, internet off)

**What this notebook does:** it reads every hidden test study from DICOM (with **the same** preprocessing as the
cache), runs the trained fold models, merges the folds with a rank mean and writes `submission.csv`.

**Why did our first kernel score 0.500?** When DICOMs could not be read or the checkpoint did not match, it silently
wrote constant prior values; a constant prediction means AUC = 0.5 for every finding.
This notebook has three guards; if any of them fires it **raises an error** (a failed submission is better than a
silent 0.5, because you notice immediately):
1. The checkpoint is loaded with `strict=True`; a key mismatch is an error.
2. More than 2% unreadable studies is an error.
3. The same prediction for every study is an error.

**Kaggle settings:** Accelerator = GPU, Internet = **off** (competition rule),
Input = competition data + `rsna-knee-src` + the output of 05 (checkpoints). Time limit 9 hours.

In [ ]:
# Add the project root (the folder that contains src/) to the Python path; works locally and on Kaggle.
import os, sys
from pathlib import Path

def add_project_root():
    cands = [Path.cwd(), Path.cwd().parent]
    if Path("/kaggle/input").exists():
        # os.walk with pruning: never enters the competition DICOM folders (rglob took minutes)
        for d, dirs, files in os.walk("/kaggle/input"):
            dirs[:] = [x for x in dirs if x not in ("train_series", "test_series")]
            if Path(d).name == "src" and "__init__.py" in files:
                cands.append(Path(d).parent)
    for c in cands:
        if (c / "src" / "__init__.py").exists():
            sys.path.insert(0, str(c))
            return c
    raise FileNotFoundError("src/ not found")

PROJECT_ROOT = add_project_root()
print("Project root:", PROJECT_ROOT)

## 0. Decoders for compressed DICOM

Some series are compressed with JPEG Lossless or JPEG 2000. pydicom cannot open them without extra packages,
and the Kaggle image does not ship them. Without the decoders those series would be skipped silently
(this happened in our first cache attempt). The wheels are inside `rsna-knee-src` (`third_party/wheels`),
so no internet is needed.

In [ ]:
from pathlib import Path
from src.config import IS_KAGGLE
from src.dicom import ensure_decoders

from src.config import find_input
WHEELS = [p for p in [find_input("third_party/wheels")] if p] if IS_KAGGLE else []
DECODERS = ensure_decoders(WHEELS)
print("DICOM decoders:", DECODERS)
if IS_KAGGLE:
    assert DECODERS["libjpeg"] and DECODERS["openjpeg"], "JPEG decoders could not be installed!"

In [ ]:
import time
from pathlib import Path

import pandas as pd
import torch

from src.cache import CacheConfig
from src.config import DATA_DIR, EXPERIMENTS_DIR, IS_KAGGLE, TARGET_COLS
from src.inference import predict_test, rank_blend, validate_submission
from src.model import load_checkpoint

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
ckpt_root = Path("/kaggle/input") if IS_KAGGLE else EXPERIMENTS_DIR
from src.config import find_inputs
CKPTS = find_inputs("model_fold*.pt", str(ckpt_root))
print("Checkpoint'ler:", *CKPTS, sep="\n  ")
if not CKPTS:
    raise FileNotFoundError("checkpoint not found: attach the output of notebook 05 as an input")

In [ ]:
models, cache_cfgs = [], set()
for p in CKPTS:
    ck = torch.load(str(p), map_location="cpu", weights_only=False)
    cache_cfgs.add(tuple(sorted(ck["cache_config"].items())))
    models.append(load_checkpoint(p, str(device)))
    print(f"{p.name}: fold {ck['fold']} epoch {ck['epoch']} | {ck['metrics']}")
assert len(cache_cfgs) == 1, "folds were trained with different preprocessing settings!"
CACHE_CFG = CacheConfig(**dict(next(iter(cache_cfgs))))
print("Preprocessing:", CACHE_CFG)

In [ ]:
test = pd.read_csv(DATA_DIR / "test.csv")
series = pd.read_csv(DATA_DIR / "test_series.csv")
sample = pd.read_csv(DATA_DIR / "sample_submission.csv")
MAX_FAIL = 0.02 if IS_KAGGLE else 1.0  # locally only 1 of the 3 test studies has DICOMs
t0 = time.time()
pred, meta = predict_test(models, test, series, DATA_DIR, CACHE_CFG, device,
                          batch_size=2, num_workers=4 if IS_KAGGLE else 0, max_fail_frac=MAX_FAIL)
print(f"Inference: {len(pred)} vaka, {(time.time() - t0) / 60:.1f} dk")
meta.reindex(columns=["StudyInstanceUID", "n_slots", "side", "side_source", "errors"]).head()

## Submission file

The folds are already merged with a rank mean (values are rank percentiles in 0–1). AUC only looks at the ranking,
so this carries the same information as a probability mean and is robust to scale differences. `rank_blend`
is used the same way to combine other models.

In [ ]:
sub = sample[["StudyInstanceUID"]].merge(pred, on="StudyInstanceUID", how="left")[["StudyInstanceUID"] + TARGET_COLS]
if IS_KAGGLE:
    validate_submission(sub, sample)
else:
    sub = sub.fillna(0.5)  # yerel deneme: DICOM'u olmayan vakalar
sub.to_csv("submission.csv", index=False)
print("submission.csv written:", sub.shape)
sub.head()